# B2-020-language-transformers — Practice p24 — Solution

**Type:** challenge · **Difficulty:** advanced · **Concepts:** nlp-pretraining-objectives, nlp-fine-tuning-protocol, transformer-nlp-task-design

*55 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260812`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C11-neural-training`, `B2-019-attention-transformers`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb).

## Solution

The earliest MLM violation is input construction, so hide selected truths before any forward pass. The earliest fine-tuning violation is optimizer construction, so include the trainable head before training. The earliest evaluation violation is split construction, so remove train indices before computing any metric—not afterward.

In [ ]:
import importlib.util

def load_literal_module(name, relative_path):
    spec = importlib.util.spec_from_file_location(name, relative_path)
    assert spec is not None and spec.loader is not None
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

import torch
from torch import nn
from torch.nn import functional as F

fixture = load_literal_module("language_fixture_p24", "../data/language_fixture.py")

def evaluation_indices(train_indices, candidate_indices):
    train = set(train_indices)
    return sorted(index for index in set(candidate_indices) if index not in train)

train_indices = list(fixture.TRAIN_SPLIT_IDS)
heldout_indices = evaluation_indices(train_indices, [2, 4, 5])
true_tokens = torch.tensor([[4,5,4], [5,4,5]], dtype=torch.int64)
selected = torch.tensor([[False,True,False], [True,False,False]])
masked_inputs = true_tokens.clone(); masked_inputs[selected] = 1
mlm_labels = torch.full_like(true_tokens, -100); mlm_labels[selected] = true_tokens[selected]

rows = torch.tensor(fixture.INTENT_INPUT_IDS, dtype=torch.int64)
labels = torch.tensor(fixture.INTENT_LABELS, dtype=torch.int64)
features = F.one_hot(rows, num_classes=12).float().sum(dim=1)[:, 4:12]
torch.manual_seed(20260812)
classifier = nn.Linear(8, 2)
head_before = {name: value.detach().clone() for name, value in classifier.state_dict().items()}
optimizer = torch.optim.AdamW(classifier.parameters(), lr=0.03, weight_decay=0)
train_tensor = torch.tensor(train_indices)
for _ in range(40):
    optimizer.zero_grad(set_to_none=True)
    loss = F.cross_entropy(classifier(features[train_tensor]), labels[train_tensor])
    loss.backward(); optimizer.step()
head_after = classifier.state_dict()
heldout_tensor = torch.tensor(heldout_indices)
heldout_accuracy = classifier(features[heldout_tensor]).argmax(1).eq(labels[heldout_tensor]).float().mean().item()

### Answer check

In [ ]:
assert heldout_indices == [4, 5]
assert set(train_indices).isdisjoint(heldout_indices)
assert torch.equal(masked_inputs[selected], torch.ones(selected.sum(), dtype=torch.int64))
assert torch.equal(mlm_labels[selected], true_tokens[selected])
assert all(value == -100 for value in mlm_labels[~selected].tolist())
assert any(not torch.equal(head_before[name], head_after[name]) for name in head_before)
assert heldout_accuracy == 1.0